In [1]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_{CitationFull}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 13:43:59,790] A new study created in memory with name: no-name-e5fe9a80-563c-4479-912e-a86c1d688339



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 14:29:07,772] Trial 0 finished with value: 0.9384858210881551 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9384858210881551.
[I 2026-09-22 14:30:21,441] Trial 1 finished with value: 0.949526826540629 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.949526826540629.
[I 2026-09-22 14:30:51,667] Trial 2 finished with value: 0.9421661496162415 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.949526826540629.
[I 2026-09-22 14:31:33,754] Trial 3 finished with value: 0.9348054726918539 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9495268

[I 2026-09-22 14:59:27,913] A new study created in memory with name: no-name-8bbc8b01-cb7d-4b2f-a3cf-5a4494d0c8c3


GCNII: 0.9539 +/- 0.0065

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 14:59:47,368] Trial 0 finished with value: 0.9463722507158915 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9463722507158915.
[I 2026-09-22 15:00:10,680] Trial 1 finished with value: 0.9490010539690653 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9490010539690653.
[I 2026-09-22 15:00:24,001] Trial 2 finished with value: 0.9353312452634176 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9490010539690653.
[I 2026-09-22 15:00:46,701] Trial 3 finished with value: 0.9616193771362305 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.961619377136

In [1]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "80:10:10":
        transform = RandomNodeSplit(split="train_rest", num_val=0.10, num_test=0.10)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "80:10:10"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_{CitationFull}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 16:22:43,963] A new study created in memory with name: no-name-7e482422-8b85-446c-bf18-d3e5c1b77ec6



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 16:31:54,869] Trial 0 finished with value: 0.9408983588218689 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9408983588218689.
[I 2026-09-22 16:32:11,644] Trial 1 finished with value: 0.9535066882769266 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9535066882769266.
[I 2026-09-22 16:32:15,316] Trial 2 finished with value: 0.9416863719622294 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.9535066882769266.
[I 2026-09-22 16:32:21,879] Trial 3 finished with value: 0.9369582533836365 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9535

[I 2026-09-22 16:39:41,338] A new study created in memory with name: no-name-294e5e77-249b-4aa5-8712-583e7d3d2e7a


GCNII: 0.9662 +/- 0.0071

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 16:39:46,320] Trial 0 finished with value: 0.9487785696983337 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9487785696983337.
[I 2026-09-22 16:39:54,008] Trial 1 finished with value: 0.9527186751365662 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9527186751365662.
[I 2026-09-22 16:39:58,924] Trial 2 finished with value: 0.94247438510259 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9527186751365662.
[I 2026-09-22 16:40:04,458] Trial 3 finished with value: 0.9661150574684143 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.96611505746841

In [2]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora_ML"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_{CitationFull}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

Optuna search for GCN


[I 2026-09-22 16:43:38,781] A new study created in memory with name: no-name-b2617627-c436-4f82-b667-f65a7cd8d1d3


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


[I 2026-09-22 16:55:54,149] Trial 0 finished with value: 0.8923533956209818 and parameters: {'hidden': 32, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8923533956209818.
[I 2026-09-22 16:55:56,419] Trial 1 finished with value: 0.8982925415039062 and parameters: {'hidden': 16, 'dropout': 0.5, 'lr': 0.005, 'weight_decay': 0.005}. Best is trial 1 with value: 0.8982925415039062.
[I 2026-09-22 16:55:58,383] Trial 2 finished with value: 0.901262104511261 and parameters: {'hidden': 64, 'dropout': 0.5, 'lr': 0.01, 'weight_decay': 0.005}. Best is trial 2 with value: 0.901262104511261.
[I 2026-09-22 16:55:59,740] Trial 3 finished with value: 0.897550125916799 and parameters: {'hidden': 64, 'dropout': 0.5, 'lr': 0.02, 'weight_decay': 0.001}. Best is trial 2 with value: 0.901262104511261.
[I 2026-09-22 16:56:00,826] Trial 4 finished with value: 0.8938381671905518 and parameters: {'hidden': 16, 'dropout': 0.3, 'lr': 0.01, 'weight_decay': 0.0005}. Best is trial 

...

[I 2026-09-22 17:11:17,463] Trial 0 finished with value: 0.8968077699343363 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8968077699343363.
[I 2026-09-22 17:11:27,837] Trial 1 finished with value: 0.8982925216356913 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8982925216356913.
[I 2026-09-22 17:11:30,641] Trial 2 finished with value: 0.901262084643046 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.901262084643046.
[I 2026-09-22 17:11:35,622] Trial 3 finished with value: 0.8968077500661215 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.901262

[I 2026-09-22 17:15:39,635] A new study created in memory with name: no-name-8e041800-71c0-4203-9455-bf8696f5d841


GCNII: 0.8960 +/- 0.0114

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 17:15:57,124] Trial 0 finished with value: 0.906458834807078 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.906458834807078.
[I 2026-09-22 17:16:21,029] Trial 1 finished with value: 0.9042316476504008 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.906458834807078.
[I 2026-09-22 17:16:37,221] Trial 2 finished with value: 0.907943606376648 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.907943606376648.
[I 2026-09-22 17:16:50,576] Trial 3 finished with value: 0.8982925216356913 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.907943606376648.


In [3]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "80:10:10":
        transform = RandomNodeSplit(split="train_rest", num_val=0.10, num_test=0.10)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora_ML"
SPLIT_TYPE = "80:10:10"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_{CitationFull}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 17:26:20,589] A new study created in memory with name: no-name-83ecee57-4cfa-44e8-9e83-267172bf052d



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 17:54:22,907] Trial 0 finished with value: 0.9066666960716248 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9066666960716248.
[I 2026-09-22 17:54:33,234] Trial 1 finished with value: 0.9077778061230978 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9077778061230978.
[I 2026-09-22 17:54:36,905] Trial 2 finished with value: 0.9100000262260437 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.9100000262260437.
[I 2026-09-22 17:54:43,849] Trial 3 finished with value: 0.9077778061230978 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.9100

[I 2026-09-22 18:00:37,353] A new study created in memory with name: no-name-87b9b6ed-42ec-4bdb-99f2-4ba9006bb01e


GCNII: 0.9020 +/- 0.0125

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 18:00:57,787] Trial 0 finished with value: 0.9144444664319357 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9144444664319357.
[I 2026-09-22 18:01:19,950] Trial 1 finished with value: 0.9155555764834086 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9155555764834086.
[I 2026-09-22 18:01:37,068] Trial 2 finished with value: 0.9177777965863546 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.9177777965863546.
[I 2026-09-22 18:01:48,149] Trial 3 finished with value: 0.9033333659172058 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.917777796586

In [4]:
#amazon

In [5]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Amazon
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Amazon(root=root, name=dataset_name)
    elif split_type == "80:10:10":
        transform = RandomNodeSplit(split="train_rest", num_val=0.10, num_test=0.10)
        dataset = Amazon(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Photo"
SPLIT_TYPE = "80:10:10"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_{CitationFull}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 18:10:21,309] A new study created in memory with name: no-name-c3f5c3b9-41c4-4d8c-bd17-ae105e6b115c



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 19:53:39,571] Trial 0 finished with value: 0.9529411792755127 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9529411792755127.
[I 2026-09-22 19:56:37,051] Trial 1 finished with value: 0.9642701546351115 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9642701546351115.
[I 2026-09-22 19:57:15,923] Trial 2 finished with value: 0.9586056669553121 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.9642701546351115.
[I 2026-09-22 19:57:58,858] Trial 3 finished with value: 0.9511982599894205 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9642

[I 2026-09-22 20:26:39,939] A new study created in memory with name: no-name-e56354d7-7877-4408-adc9-388399bd5237


GCNII: 0.9548 +/- 0.0047

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 20:28:03,991] Trial 0 finished with value: 0.9525054494539896 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9525054494539896.
[I 2026-09-22 20:30:29,373] Trial 1 finished with value: 0.9607843160629272 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9607843160629272.
[I 2026-09-22 20:31:27,908] Trial 2 finished with value: 0.9538126389185587 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9607843160629272.
[I 2026-09-22 20:32:55,994] Trial 3 finished with value: 0.957298477490743 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9607843160629

In [6]:
 ! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Amazon
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Amazon(root=root, name=dataset_name)
    elif split_type == "70:15:15":
        transform = RandomNodeSplit(split="train_rest", num_val=0.15, num_test=0.15)
        dataset = Amazon(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Photo"
SPLIT_TYPE = "70:15:15"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 21:18:22,484] A new study created in memory with name: no-name-67d4caac-904a-4c26-af67-4d07f6ca49d0



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 23:08:21,789] Trial 0 finished with value: 0.9520906209945679 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9520906209945679.
[I 2026-09-22 23:11:47,762] Trial 1 finished with value: 0.9625435670216879 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9625435670216879.
[I 2026-09-22 23:12:25,650] Trial 2 finished with value: 0.9549942016601562 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.9625435670216879.
[I 2026-09-22 23:13:09,764] Trial 3 finished with value: 0.9430894255638123 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9625

[I 2026-09-23 00:24:41,947] A new study created in memory with name: no-name-f9aedd95-e259-4863-a08e-8b15d148a702


GCNII: 0.9524 +/- 0.0046

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-23 00:26:20,144] Trial 0 finished with value: 0.9555749297142029 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9555749297142029.
[I 2026-09-23 00:28:13,415] Trial 1 finished with value: 0.9544134736061096 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9555749297142029.
[I 2026-09-23 00:29:10,193] Trial 2 finished with value: 0.9529616634051005 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9555749297142029.
[I 2026-09-23 00:30:33,221] Trial 3 finished with value: 0.957026739915212 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.9570267399152